In [27]:
import json
from itertools import combinations
from datasketch import MinHash, MinHashLSH
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import networkx as nx
from huggingface_hub import hf_hub_download



In [19]:

# ==========================================
# 0. DATA LOADING & EXTRACTION
# ==========================================

def load_texts_from_jsonl(filepath):
    texts = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            if not line.strip():
                continue
            data = json.loads(line)

            # Extract content from all messages in the record
            # Adjust indexing if you only want a specific message index
            for msg_idx, message in enumerate(data.get("messages", [])):
                content = message.get("content", "").strip()
                if content:
                    texts.append(content)

    print(f"Loaded {len(texts)} texts from JSONL.")
    return texts


# ==========================================
# 1. TF-IDF + COSINE SIMILARITY
# ==========================================


def find_similar_tfidf(texts, threshold=0.75):
    """Best for: Finding topically similar texts or shared vocabulary."""
    print(f"\n--- 1. Running TF-IDF Cosine Similarity (Threshold >= {threshold}) ---")

    # Build TF-IDF matrix using 1- and 2-grams
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2), min_df=2, stop_words="english"
    )
    tfidf_matrix = vectorizer.fit_transform(texts)

    # Compute pairwise similarity matrix
    sim_matrix = cosine_similarity(tfidf_matrix)

    similar_pairs = []
    num_texts = len(texts)

    # Extract upper triangle of the similarity matrix (avoid duplicate pairs & self-similarity)
    for i in range(num_texts):
        for j in range(i + 1, num_texts):
            score = sim_matrix[i, j]
            if score >= threshold:
                similar_pairs.append((i, j, score))

    # Sort pairs by similarity descending
    similar_pairs.sort(key=lambda x: x[2], reverse=True)

    print(f"Found {len(similar_pairs)} TF-IDF similar pairs.")
    return similar_pairs


# ==========================================
# 2. MINHASH + LSH (Near-Duplicates)
# ==========================================


def get_shingles(text, k=3):
    """Helper: Break text into character or word N-grams (shingles)."""
    words = text.lower().split()
    if len(words) < k:
        return {" ".join(words)}
    return {" ".join(words[i : i + k]) for i in range(len(words) - k + 1)}


def find_similar_minhash_lsh(
    texts, threshold=0.80, num_perm=128, shingle_size=3):
    """Best for: Fast near-duplicate detection at scale."""
    print(f"\n--- 2. Running MinHash LSH (Jaccard Threshold >= {threshold}) ---")

    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    minhashes = {}

    # Create MinHash signatures and insert into LSH index
    for idx, text in enumerate(texts):
        m = MinHash(num_perm=num_perm)
        for shingle in get_shingles(text, k=shingle_size):
            m.update(shingle.encode("utf8"))

        lsh.insert(f"doc_{idx}", m)
        minhashes[idx] = m

    similar_pairs = set()

    # Query LSH for each document
    for idx in range(len(texts)):
        result = lsh.query(minhashes[idx])
        for match_id in result:
            match_idx = int(match_id.split("_")[1])
            if match_idx > idx:  # Avoid duplicates and self-matching
                # Estimate actual Jaccard score from MinHash signatures
                score = minhashes[idx].jaccard(minhashes[match_idx])
                similar_pairs.add((idx, match_idx, score))

    sorted_pairs = sorted(list(similar_pairs), key=lambda x: x[2], reverse=True)
    print(f"Found {len(sorted_pairs)} MinHash near-duplicate pairs.")
    return sorted_pairs



# ==========================================
# 3. JACCARD N-GRAM OVERLAP (Exact Set Math)
# ==========================================


def compute_jaccard(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union


def find_similar_jaccard(texts, threshold=0.70, shingle_size=3):
    """Best for: Exact set-overlap similarity on short texts without vectorization."""
    print(f"\n--- 3. Running Jaccard N-gram Overlap (Threshold >= {threshold}) ---")

    # Precompute shingle sets for all texts
    shingle_sets = [get_shingles(t, k=shingle_size) for t in texts]

    similar_pairs = []

    # Pairwise comparison across the dataset
    for i, j in combinations(range(len(texts)), 2):
        score = compute_jaccard(shingle_sets[i], shingle_sets[j])
        if score >= threshold:
            similar_pairs.append((i, j, score))

    similar_pairs.sort(key=lambda x: x[2], reverse=True)
    print(f"Found {len(similar_pairs)} Jaccard similar pairs.")
    return similar_pairs


def get_texts_to_replace_from_pairs(texts, tfidf_pairs, threshold=0.85):
    """Takes a list of (i, j, score) pairs and returns unique text indices

    to replace, keeping the longest text from each similar group.
    """
    # 1. Build a graph where edges connect similar texts above the threshold
    G = nx.Graph()
    G.add_nodes_from(range(len(texts)))

    for i, j, score in tfidf_pairs:
        if score >= threshold:
            G.add_edge(i, j)

    # 2. Find connected groups (clusters of similar texts)
    indices_to_replace = set()
    clusters_found = 0

    for component in nx.connected_components(G):
        if len(component) > 1:
            clusters_found += 1
            # Sort cluster members by text length (descending) to keep the richest/longest text
            sorted_members = sorted(
                list(component), key=lambda idx: len(texts[idx]), reverse=True
            )

            # Keep the first one (longest), mark the rest for replacement
            indices_to_replace.update(sorted_members[1:])

    print(
        f"Found {clusters_found} redundant clusters at threshold >= {threshold}."
    )
    print(
        f"Total unique texts flagged for replacement: {len(indices_to_replace)}"
    )

    return sorted(list(indices_to_replace))


def filter_and_save_jsonl(input_filepath, output_filepath, indices_to_delete):
    """Reads a JSONL file, removes the specified line indices,

    and saves the cleaned records to a new JSONL file.
    """
    # Convert list to a set for O(1) instant lookups
    delete_set = set(indices_to_delete)

    kept_count = 0
    deleted_count = 0

    with open(input_filepath, "r", encoding="utf-8") as infile, open(
        output_filepath, "w", encoding="utf-8"
    ) as outfile:

        for line_idx, line in enumerate(infile):
            if not line.strip():
                continue

            # If the index is flagged, skip writing it
            if line_idx in delete_set:
                deleted_count += 1
                continue

            # Otherwise, write the line unchanged to the new file
            outfile.write(line)
            kept_count += 1

In [ ]:
# import json

# with open("./jsonl_files/dataset.jsonl", "r", encoding="utf-8") as file:
#     data = [json.loads(line) for line in file if line.strip()]



In [3]:
texts = load_texts_from_jsonl("./jsonl_files/dataset.jsonl")

Loaded 3300 texts from JSONL.


In [10]:

# 1. TF-IDF Cosine Similarity
tfidf_pairs = find_similar_tfidf(texts, threshold=0.85)

# 2. MinHash LSH
minhash_pairs = find_similar_minhash_lsh(texts, threshold=0.75)

# 3. Exact Jaccard N-gram Overlap
jaccard_pairs = find_similar_jaccard(texts, threshold=0.65)




--- 1. Running TF-IDF Cosine Similarity (Threshold >= 0.85) ---
Found 720 TF-IDF similar pairs.

--- 2. Running MinHash LSH (Jaccard Threshold >= 0.75) ---
Found 1 MinHash near-duplicate pairs.

--- 3. Running Jaccard N-gram Overlap (Threshold >= 0.65) ---
Found 5 Jaccard similar pairs.


In [15]:
replace_indices = get_texts_to_replace_from_pairs(texts, tfidf_pairs, threshold=0.85)


Found 590 redundant clusters at threshold >= 0.85.
Total unique texts flagged for replacement: 671


In [20]:
INPUT_JSONL = "./jsonl_files/dataset.jsonl"
OUTPUT_JSONL = "./jsonl_files/dataset_normal.jsonl"

filter_and_save_jsonl(INPUT_JSONL, OUTPUT_JSONL, replace_indices)

In [21]:
texts_normal= load_texts_from_jsonl("./jsonl_files/dataset_normal.jsonl")


Loaded 2629 texts from JSONL.


In [22]:

# 1. TF-IDF Cosine Similarity
tfidf_pairs = find_similar_tfidf(texts_normal, threshold=0.85)

# 2. MinHash LSH
minhash_pairs = find_similar_minhash_lsh(texts_normal, threshold=0.75)

# 3. Exact Jaccard N-gram Overlap
jaccard_pairs = find_similar_jaccard(texts_normal, threshold=0.65)



--- 1. Running TF-IDF Cosine Similarity (Threshold >= 0.85) ---
Found 26 TF-IDF similar pairs.

--- 2. Running MinHash LSH (Jaccard Threshold >= 0.75) ---
Found 0 MinHash near-duplicate pairs.

--- 3. Running Jaccard N-gram Overlap (Threshold >= 0.65) ---
Found 0 Jaccard similar pairs.


In [24]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
)
hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]


In [ ]:
repo_id = "daniloreddy/Qwen3.5-9B_GGUF"
filename = "Qwen3.5-9B_Q4_K_M.gguf"

model_dir = Path(cache_dir) / "qwen3.5-9b"
model_dir.mkdir(parents=True, exist_ok=True)


model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    token=hf_token,          
    local_dir=model_dir
)


In [36]:

class LlamaServerQueryGenerator:
    def __init__(
        self,
        system_prompt,
        base_url="http://127.0.0.1:8080/v1",
        max_tokens=512,
        temperature=0.7,
    ):
        self.client = OpenAI(
            base_url=base_url,
            api_key="not-needed",
        )
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {
                "role": "system",
                "content": self.system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

    def generate(self, user_prompt):
        response = self.client.chat.completions.create(
            model="local-model",
            messages=self.build_messages(user_prompt),
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            top_p=0.8,
            extra_body={
                "top_k": 20,
                "min_p": 0,
            },
        )

        return response.choices[0].message.content

In [ ]:
SYSTEM_PROMPT ="""
You are an expert in rewriting a text without changing the vital informations. 
You resposibillty is to change the user input text into something differetn fom vocabulary and grammer POVs. 
You MUST not chnage the important and vital informations suchas number, dates or Names.
These inputs are about asking for stock prices of asome companies from custom periods. You Do not need to follwo the user input. Just rephrase the user input. 
You colleage will use your output for tool calling later. 
When you rephrase you must make it more complex.
"""

In [64]:
query_generator_llama_cpp = LlamaServerQueryGenerator(
    system_prompt=SYSTEM_PROMPT,
    max_tokens=256,
    temperature=0.7,
)

In [67]:
user_prompt =  "I need the weekly price data for Cisco Systems covering the period from January 2022 to May 2025."

In [68]:
model_output = query_generator_llama_cpp.generate(user_prompt)
model_output

'Please retrieve the comprehensive weekly stock price dataset for Cisco Systems, specifically encompassing the temporal interval extending from the onset of January 2022 through the conclusion of May 2025.'